## Complete Transformer Block in a Transformer

Notebook implements a full **pre-norm** Transformer block, where the pipeline is roughly:
````text
           Input X
              │
              ├─────────────────────────┐
              │                         │
              ↓                         │
           RMSNorm                      │
              │                         │ Residual Path
              ↓                         │
Causal Multi-Head Self-Attention        │
              │                         │
              └─────────→ (+) ←─────────┘
                           │
                           ↓
            Contextualized Representation X'
                           │
      ┌────────────────────┴────────────────────┐
      │                                         │
      ↓                                         │
   RMSNorm                                      │
      │                                         │ Residual Path
      ↓                                         │
 SwiGLU MLP                                     │
      │                                         │
      ├─→ Gate Proj → SiLU ──┐                  ↓
      │                      ⊙ ─→ Down Proj ─→ (+)
      └────→ Value Proj ─────┘                  │
                                                ↓
                                             Output Y
````

Where the input tensor shape is preserved $X:[B,T,d_{model}] \rightarrow Y:[B,T,d_{model}]$

---

The entire pipeline is commonly denoted as:

$X' = X + \operatorname{Attention}(\operatorname{RMSNorm}(X))$ \
$Y = X' + \operatorname{MLP}(\operatorname{RMSNorm}(X'))$

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[1] # cwd up two directories
sys.path.append(str(ROOT))

In [ ]:
import torch

from src.llm.config import ModelConfig
from src.llm.attention import CausalSelfAttention
from src.llm.layers import RMSNorm, SwiGLU

## Initialize Configs

Low-scale, practical toy dimensions are to be used.

In [ ]:
config = ModelConfig(
    vocab_size=1000, # not used in this example
    d_model=8,
    num_heads=2,
    d_ff=32,
    num_layers=1
)

Then the main block sublayers. Notice the two **separate** normalizations `attn_norm` and `mlp_norm`. \
Same class objects, however they have distinct, **independently** learned parameters

In [ ]:
# Attention sublayer
attn_norm = RMSNorm(config.d_model)
attention = CausalSelfAttention(config)

# Feedforward sublayer
mlp_norm = RMSNorm(config.d_model)
mlp = SwiGLU(config)

In [ ]:
# Input
x = torch.randn(
    1,
    4,
    config.d_model
)

print("Input:", x.shape)

## Attention Sublayer
The **self-attention** sublayer normalizes the input $X$, applies self-attention, then adds the contextualized representation back to the original input through a **residual connection**.

````text
           Input X
              │
              ├─────────────────────────┐
              │                         │
              ↓                         │
           RMSNorm                      │
              │                         │ Residual Path
              ↓                         │
Causal Multi-Head Self-Attention        │
              │                         │
              └─────────→ (+) ←─────────┘
                  Residual Connection
````

---

Or simply:

$X' = X + \operatorname{Attention}(\operatorname{RMSNorm}(X))$

In [ ]:
# First normalization
normalized = attn_norm(x)

# Self-attention
attn_output = attention(normalized)

# Add residual
x = x + attn_output # old representation + contextualized representation

print("After attention residual:", x.shape)

## Feedforward/MLP Sublayer
The attention sublayer's output $X'$ passes through the **MLP** sublayer, which normalizes $X'$, applies the SwiGLU transformation, then adds the transformed representation back to the original through a **second residual connection**.
````text
            Input from Attention Sublayer X'                
                           │
      ┌────────────────────┴────────────────────┐
      │                                         │
      ↓                                         │
   RMSNorm                                      │
      │                                         │ Residual Path
      ↓                                         │
 SwiGLU MLP                                     │
      │                                         │
      ├─→ Gate Proj → SiLU ──┐                  ↓
      │                      ⊙ ─→ Down Proj ─→ (+) Residual Connection
      └────→ Value Proj ─────┘                  │
                                                ↓
                                             Output Y
````

---

Or simply:

$Y = X' + \operatorname{MLP}(\operatorname{RMSNorm}(X'))$

In [ ]:
# Second normalization
normalized = mlp_norm(x)

# Feedforward network
mlp_output = mlp(normalized)

x = x + mlp_output # previous representation + transformed representation

print("After MLP residual:", x.shape)

Notice the tensor shape is **preserved**: \
$X:[B,T,d_{model}] \rightarrow \text{Transformer block} \rightarrow Y:[B,T,d_{model}]$

This is crucial as that **shape preservation** is what allows us to stack many Transformer blocks later.

---

Conceptually: \
$Y_1$ from the **current** block becomes the input $X_2$ for the **next** block, which outputs $Y_2$, which becomes $X_3$ and so on...